# Chapter 13: Data Association

<a href="../lite/lab/index.html?path=ch13_data_association.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_covariance_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    e = Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(e)
    return e

A robot sees two trees. Its map has three trees. Which measurement goes with which tree?
Get it right and the robot refines its position. Get it wrong and the map tears itself apart.
This seemingly simple matching problem is the number one cause of SLAM failures in the real world.

**Data association** is the problem of assigning measurements to map features (or to "new feature").
Every estimation algorithm assumes this assignment is correct. When it is wrong, the estimator
incorporates contradictory information and the result is catastrophic.

## 13.1 Correspondence Problem

Given:
- A set of **predicted measurements** $\hat{z}_1, \hat{z}_2, \ldots, \hat{z}_M$ (one per map landmark)
- A set of **actual measurements** $z_1, z_2, \ldots, z_K$

Find the assignment: which $z_k$ corresponds to which $\hat{z}_m$?

This is combinatorial. With $K$ measurements and $M$ landmarks, there are $M^K$ possible assignments.
For 10 measurements and 50 landmarks, that is $\approx 10^{17}$ combinations.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_landmarks = 6           # number of landmarks in the map
n_measurements = 4         # number of measurements received
sensor_noise = 0.5         # measurement noise std (meters)
# ──────────────────────────────────────────────────────────────────────────────

# True landmark positions
landmarks = np.array([[2, 3], [5, 1], [7, 4], [3, 7], [8, 6], [1, 5]])[:n_landmarks]

# Robot at origin, observes some landmarks with noise
robot_pos = np.array([4.0, 4.0])
visible_idx = np.random.choice(n_landmarks, n_measurements, replace=False)
measurements = landmarks[visible_idx] - robot_pos + np.random.normal(0, sensor_noise, (n_measurements, 2))

# Predicted measurements (from all landmarks)
predicted = landmarks - robot_pos

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: the problem
ax = axes[0]
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=100, marker='^', zorder=5, label='map landmarks')
for i, lm in enumerate(landmarks):
    ax.annotate(f'L{i}', xy=lm, xytext=(5, 5), textcoords='offset points', fontsize=10)
ax.plot(*robot_pos, 'ko', ms=10, zorder=5)
ax.annotate('Robot', xy=robot_pos, xytext=(5, -15), textcoords='offset points', fontsize=11, fontweight='bold')
# Show measurements as relative vectors from robot
for k, z in enumerate(measurements):
    meas_pos = robot_pos + z
    ax.scatter(*meas_pos, c='tomato', s=60, marker='x', zorder=5)
    ax.annotate(f'z{k}', xy=meas_pos, xytext=(5, 5), textcoords='offset points', color='tomato')
ax.set_title("Which measurement matches which landmark?", fontsize=13)
ax.set_aspect('equal'); ax.legend()

# Right: correct assignment
ax = axes[1]
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=100, marker='^', zorder=5)
ax.plot(*robot_pos, 'ko', ms=10, zorder=5)
for k, z in enumerate(measurements):
    meas_pos = robot_pos + z
    true_lm = landmarks[visible_idx[k]]
    ax.scatter(*meas_pos, c='tomato', s=60, marker='x', zorder=5)
    ax.plot([meas_pos[0], true_lm[0]], [meas_pos[1], true_lm[1]], 'g-', lw=2, alpha=0.7)
ax.set_title("Correct data association (ground truth)", fontsize=13)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 13.2 Nearest Neighbor

The simplest strategy: assign each measurement to the **closest predicted measurement**.

$$c_k = \arg\min_m \| z_k - \hat{z}_m \|$$

This works well when landmarks are well separated and noise is small.
It fails when landmarks are close together or noise is large.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(7)
n_landmarks = 8
sensor_noise = 0.3        # try 0.3 (easy) vs 1.5 (hard)
robot_pos = np.array([5.0, 5.0])
# ──────────────────────────────────────────────────────────────────────────────

landmarks = np.random.uniform(1, 9, (n_landmarks, 2))
visible_idx = np.random.choice(n_landmarks, 5, replace=False)
measurements = landmarks[visible_idx] - robot_pos + np.random.normal(0, sensor_noise, (5, 2))
predicted = landmarks - robot_pos

# Nearest neighbor assignment
associations = []
for k, z in enumerate(measurements):
    dists = np.linalg.norm(predicted - z, axis=1)
    best = np.argmin(dists)
    associations.append(best)

# Check correctness
correct = sum(a == t for a, t in zip(associations, visible_idx))

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=100, marker='^', zorder=5, label='landmarks')
ax.plot(*robot_pos, 'ko', ms=10, zorder=5)
for k, z in enumerate(measurements):
    meas_pos = robot_pos + z
    assoc_lm = landmarks[associations[k]]
    true_lm = landmarks[visible_idx[k]]
    color = 'forestgreen' if associations[k] == visible_idx[k] else 'tomato'
    ax.scatter(*meas_pos, c='orange', s=60, marker='x', zorder=5)
    ax.plot([meas_pos[0], assoc_lm[0]], [meas_pos[1], assoc_lm[1]], color=color, lw=2, alpha=0.8)

ax.set_title(f"Nearest Neighbor: {correct}/{len(measurements)} correct (noise σ={sensor_noise})", fontsize=13)
ax.set_aspect('equal')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Associations: {associations}")
print(f"True matches: {list(visible_idx)}")
print(f"Correct: {correct}/{len(measurements)}")

**Key observations:**
- Nearest neighbor is fast: $O(KM)$ for $K$ measurements and $M$ landmarks.
- It is **greedy** and can fail when landmarks are clustered or noise is large.
- It does not account for measurement uncertainty (a noisy sensor and a precise sensor are treated the same).

## 13.3 Gating

Before even considering a match, we can **reject impossible associations** using a gate.
The **Mahalanobis distance** accounts for the shape of the uncertainty:

$$d^2_M = (z - \hat{z})^T S^{-1} (z - \hat{z})$$

where $S$ is the innovation covariance. We reject matches where $d^2_M > \chi^2_\alpha$ (the chi-squared threshold).

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
gate_probability = 0.95     # chi-squared gate (try 0.9, 0.95, 0.99)
sigma_x = 0.3; sigma_y = 0.6; rho = 0.4   # measurement covariance shape
# ──────────────────────────────────────────────────────────────────────────────

S = np.array([[sigma_x**2, rho*sigma_x*sigma_y],
              [rho*sigma_x*sigma_y, sigma_y**2]])
z_predicted = np.array([0.0, 0.0])

# Chi-squared threshold for 2 DOF
gate_threshold = chi2.ppf(gate_probability, df=2)

# Generate candidate measurements
np.random.seed(42)
candidates = np.random.uniform(-2, 2, (30, 2))
mahal_dists = np.array([c @ np.linalg.inv(S) @ c for c in candidates])
inside_gate = mahal_dists < gate_threshold

fig, ax = plt.subplots(figsize=(8, 7))

# Draw gate ellipse
draw_covariance_ellipse(ax, z_predicted, S, n_std=np.sqrt(gate_threshold),
                        fill=True, facecolor='steelblue', alpha=0.1, edgecolor='steelblue', lw=2, label=f'{gate_probability:.0%} gate')

ax.scatter(candidates[inside_gate, 0], candidates[inside_gate, 1],
           c='forestgreen', s=50, zorder=5, label='inside gate (consider)')
ax.scatter(candidates[~inside_gate, 0], candidates[~inside_gate, 1],
           c='tomato', s=50, marker='x', zorder=5, label='outside gate (reject)')
ax.plot(0, 0, 'k+', ms=15, mew=3, zorder=10)

ax.set_title(f"Mahalanobis gating ({gate_probability:.0%} confidence)", fontsize=13)
ax.set_xlabel("Innovation x"); ax.set_ylabel("Innovation y")
ax.set_aspect('equal'); ax.legend()
plt.tight_layout()
plt.show()

print(f"Gate threshold (χ² with 2 DOF, {gate_probability:.0%}): {gate_threshold:.2f}")
print(f"Inside gate: {inside_gate.sum()}/{len(candidates)}")

## 13.4 Ambiguity

Sometimes two landmarks are equally plausible matches for a measurement. The nearest
neighbor picks one arbitrarily. A better strategy is to **detect ambiguity** and defer the decision.

The **ambiguity ratio** compares the best and second best Mahalanobis distances:

$$\text{ratio} = \frac{d^2_{\text{best}}}{d^2_{\text{second}}}$$

If the ratio is close to 1, the match is ambiguous and should be rejected.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
ambiguity_threshold = 0.7   # reject if ratio > this (try 0.5, 0.7, 0.9)
n_landmarks = 10
sensor_noise = 0.5
# ──────────────────────────────────────────────────────────────────────────────

landmarks = np.random.uniform(0, 10, (n_landmarks, 2))
robot_pos = np.array([5.0, 5.0])
visible_idx = np.random.choice(n_landmarks, 6, replace=False)
measurements = landmarks[visible_idx] - robot_pos + np.random.normal(0, sensor_noise, (6, 2))
predicted = landmarks - robot_pos

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=100, marker='^', zorder=5)
ax.plot(*robot_pos, 'ko', ms=10, zorder=5)

n_accepted = 0; n_rejected = 0
for k, z in enumerate(measurements):
    dists = np.linalg.norm(predicted - z, axis=1)
    sorted_d = np.sort(dists)
    ratio = sorted_d[0] / max(sorted_d[1], 1e-10)
    best_idx = np.argmin(dists)
    meas_pos = robot_pos + z

    if ratio < ambiguity_threshold:
        color = 'forestgreen'
        ax.plot([meas_pos[0], landmarks[best_idx, 0]], [meas_pos[1], landmarks[best_idx, 1]],
                color=color, lw=2, alpha=0.7)
        n_accepted += 1
    else:
        color = 'tomato'
        n_rejected += 1
    ax.scatter(*meas_pos, c=color, s=60, marker='x', zorder=5)
    ax.annotate(f'{ratio:.2f}', xy=meas_pos, xytext=(5, 5), textcoords='offset points', fontsize=9, color=color)

ax.set_title(f"Ambiguity test: {n_accepted} accepted, {n_rejected} rejected (threshold={ambiguity_threshold})", fontsize=12)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 13.5 Outlier Rejection

Even with gating and ambiguity tests, some wrong associations slip through.
**RANSAC** (Random Sample Consensus) is a powerful tool: randomly pick minimal subsets,
fit a model, count how many measurements agree, keep the best.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_inliers = 20
n_outliers = 8
inlier_noise = 0.15
ransac_iterations = 50
inlier_threshold = 0.5
# ──────────────────────────────────────────────────────────────────────────────

# True transform: rotation + translation
true_angle = np.radians(15)
true_t = np.array([2.0, 1.0])
R_true = np.array([[np.cos(true_angle), -np.sin(true_angle)],
                    [np.sin(true_angle), np.cos(true_angle)]])

# Source points
src = np.random.uniform(0, 5, (n_inliers + n_outliers, 2))
dst = np.zeros_like(src)
dst[:n_inliers] = (R_true @ src[:n_inliers].T).T + true_t + np.random.normal(0, inlier_noise, (n_inliers, 2))
dst[n_inliers:] = np.random.uniform(0, 8, (n_outliers, 2))  # random outliers

# RANSAC
best_inliers = []
for _ in range(ransac_iterations):
    idx = np.random.choice(len(src), 2, replace=False)
    # Estimate transform from 2 points
    s1, s2 = src[idx]
    d1, d2 = dst[idx]
    ds = s2 - s1; dd = d2 - d1
    angle_est = np.arctan2(dd[1], dd[0]) - np.arctan2(ds[1], ds[0])
    R_est = np.array([[np.cos(angle_est), -np.sin(angle_est)],
                      [np.sin(angle_est), np.cos(angle_est)]])
    t_est = d1 - R_est @ s1
    # Count inliers
    transformed = (R_est @ src.T).T + t_est
    errors = np.linalg.norm(transformed - dst, axis=1)
    inlier_mask = errors < inlier_threshold
    if inlier_mask.sum() > len(best_inliers):
        best_inliers = np.where(inlier_mask)[0]
        best_R, best_t = R_est, t_est

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(src[:, 0], src[:, 1], c='steelblue', s=40, label='source')
ax.scatter(dst[:n_inliers, 0], dst[:n_inliers, 1], c='forestgreen', s=40, label='inliers')
ax.scatter(dst[n_inliers:, 0], dst[n_inliers:, 1], c='tomato', s=40, marker='x', label='outliers')
for i in range(len(src)):
    ax.plot([src[i,0], dst[i,0]], [src[i,1], dst[i,1]], 'gray', alpha=0.2)
ax.set_title("All correspondences (including outliers)", fontsize=12)
ax.set_aspect('equal'); ax.legend()

ax = axes[1]
transformed = (best_R @ src.T).T + best_t
for i in best_inliers:
    ax.plot([transformed[i,0], dst[i,0]], [transformed[i,1], dst[i,1]], 'forestgreen', lw=1.5)
ax.scatter(transformed[:, 0], transformed[:, 1], c='steelblue', s=40, label='transformed source')
ax.scatter(dst[:, 0], dst[:, 1], c='orange', s=20, alpha=0.5, label='destination')
ax.set_title(f"RANSAC result: {len(best_inliers)} inliers found", fontsize=12)
ax.set_aspect('equal'); ax.legend()

plt.tight_layout()
plt.show()

print(f"True inliers: {n_inliers}, RANSAC found: {len(best_inliers)}")
print(f"Angle error: {np.degrees(np.arctan2(best_R[1,0], best_R[0,0]) - true_angle):.2f}°")
print(f"Translation error: {np.linalg.norm(best_t - true_t):.3f} m")

## 13.6 Impact on Estimation

A single wrong data association can corrupt the entire map. The estimator "trusts" the
wrong association and distorts the state to accommodate contradictory information.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_landmarks = 5
n_observations = 20
wrong_associations = 2    # number of deliberately wrong associations (try 0, 1, 3, 5)
# ──────────────────────────────────────────────────────────────────────────────

# True landmark positions
true_landmarks = np.array([[2,3],[5,1],[8,4],[3,7],[7,7]])[:n_landmarks]

# Robot at known position, takes noisy observations
robot_pos = np.array([5.0, 4.0])
obs = []
for i in range(n_observations):
    lm_idx = np.random.randint(n_landmarks)
    z = true_landmarks[lm_idx] - robot_pos + np.random.normal(0, 0.3, 2)
    obs.append((lm_idx, z))

# Corrupt some associations
for i in range(min(wrong_associations, n_observations)):
    orig_idx = obs[i][0]
    wrong_idx = (orig_idx + 1) % n_landmarks
    obs[i] = (wrong_idx, obs[i][1])

# Simple estimation: average observations per landmark
estimated = {}
for lm_idx, z in obs:
    if lm_idx not in estimated:
        estimated[lm_idx] = []
    estimated[lm_idx].append(robot_pos + z)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, title, show_error in [(axes[0], "True landmarks", False), (axes[1], f"Estimated ({wrong_associations} wrong assoc.)", True)]:
    ax.scatter(true_landmarks[:, 0], true_landmarks[:, 1], c='steelblue', s=150, marker='^', zorder=5, label='true')
    ax.plot(*robot_pos, 'ko', ms=10, zorder=5)
    if show_error:
        for lm_idx, positions in estimated.items():
            est_pos = np.mean(positions, axis=0)
            ax.scatter(*est_pos, c='tomato', s=100, marker='s', zorder=5)
            ax.plot([true_landmarks[lm_idx, 0], est_pos[0]],
                    [true_landmarks[lm_idx, 1], est_pos[1]], 'tomato', lw=1.5, ls='--')
    ax.set_xlim(0, 10); ax.set_ylim(0, 9)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=13)

plt.suptitle("Wrong data association corrupts landmark estimates", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key observations:**
- Data association errors are **catastrophic** for estimation. A single wrong match can ruin the map.
- **Gating** prevents obviously wrong matches. **Ambiguity testing** catches uncertain ones.
- **RANSAC** provides robustness by finding the largest consistent subset.
- In practice, SLAM systems combine all three: gate first, then nearest neighbor with ambiguity check, then RANSAC for geometric verification.

---

## Exercises

### Exercise 13.1: Nearest Neighbor with Mahalanobis distance

Modify the nearest neighbor algorithm to use Mahalanobis distance instead of Euclidean distance.
Given a measurement covariance $S$, the Mahalanobis distance is $d_M = \sqrt{(z - \hat{z})^T S^{-1} (z - \hat{z})}$.
Test on a scenario where the covariance is elongated (sigma_x = 0.2, sigma_y = 2.0).
Show that Mahalanobis gives a different (and better) association than Euclidean.

In [ ]:
# Your code here
S = np.array([[0.04, 0], [0, 4.0]])  # elongated covariance
# Compute Mahalanobis: d_M = sqrt((z-z_hat).T @ inv(S) @ (z-z_hat))

### Exercise 13.2: Gating with chi-squared test

Implement Mahalanobis gating for a robot observing 8 landmarks.
Use a 95% chi-squared gate (threshold = 5.99 for 2 DOF).
Count how many spurious measurements are correctly rejected.

In [ ]:
# Your code here
gate_threshold = chi2.ppf(0.95, df=2)
# For each measurement, compute Mahalanobis distance to each landmark
# Accept only if d_M^2 < gate_threshold

### Exercise 13.3: RANSAC for 2D point matching

Implement RANSAC to estimate a rigid transform (rotation + translation) from 30 point
correspondences where 10 are outliers. Plot inliers in green and outliers in red.
Report the estimated transform error.

In [ ]:
# Your code here
# Generate inlier correspondences with known transform
# Add 10 random outlier correspondences
# Run RANSAC: sample 2 points, estimate R and t, count inliers

### Exercise 13.4: Impact analysis (challenge)

Build a simple 1D SLAM scenario: 5 poses along a line with 3 landmarks.
Each pose observes nearby landmarks. Run least squares estimation with:
(a) all correct associations, (b) one wrong association.
Compare the estimated landmark positions. How much error does one wrong association introduce?

In [ ]:
# Your code here
# Poses at x = 0, 2, 4, 6, 8
# Landmarks at x = 1, 5, 7
# Build observation equations: z = landmark - pose + noise
# Swap one association and re-solve